In [3]:
import yfinance as yf
import pandas as pd
import os
import time

# Lista spółek (Yahoo używa prostych symboli, bez .US)
# Dla klasy B Berkshire Hathaway w Yahoo to "BRK-B"
SYMBOLS = [
    "AAPL", "AMZN", "NVDA", "INTC", "AMD", "BRK-B"
]

final_data = []

print(f"--- Pobieranie fundamentów z Yahoo Finance dla {len(SYMBOLS)} spółek ---")

for sym in SYMBOLS:
    print(f"Pobieranie dla: {sym}...")

    try:
        # 1. Pobieramy obiekt tickera
        ticker = yf.Ticker(sym)

        # 2. Pobieramy dane finansowe (income statement)
        # To zwraca tabelkę DataFrame, gdzie kolumny to daty
        financials = ticker.financials

        if financials.empty:
            print(f"   ! Puste dane dla {sym}")
            continue

        # 3. Wyciągamy interesujące nas wiersze
        # Yahoo często zmienia nazwy wierszy, szukamy standardowych
        try:
            # Transpozycja (.T), żeby daty były wierszami
            df_fin = financials.T

            # Resetujemy indeks, żeby data stała się kolumną
            df_fin = df_fin.reset_index()
            df_fin = df_fin.rename(columns={"index": "report_date"})

            # Wybieramy tylko to co nas interesuje (używając .get żeby się nie wywaliło jak nie ma kolumny)
            # Uwaga: w yfinance kolumny mogą się nazywać "EBITDA", "Net Income", "Total Revenue"

            for index, row in df_fin.iterrows():
                # Yahoo zwraca daty jako Timestamp, zamieniamy na string
                report_date = str(row['report_date']).split(" ")[0]

                final_data.append({
                    "symbol": sym.replace("-", "."), # Zamieniamy BRK-B na BRK.B dla zgodności z Finnhubem
                    "report_date": report_date,
                    "year": report_date[:4],
                    "ebitda": row.get('EBITDA') or row.get('Normalized EBITDA') or 0,
                    "net_income": row.get('Net Income') or 0,
                    "revenue": row.get('Total Revenue') or 0
                })

            print(f"   -> Pobrano dane.")

        except Exception as e:
            print(f"   ! Problem z przetwarzaniem danych dla {sym}: {e}")

    except Exception as e:
        print(f"   ! BŁĄD ogólny przy {sym}: {e}")

    # Małe opóźnienie dla bezpieczeństwa
    time.sleep(1)

# --- ZAPIS ---
os.makedirs("fundamentals_data", exist_ok=True)

if final_data:
    df = pd.DataFrame(final_data)
    output_file = "fundamentals_data/company_earnings.csv"
    df.to_csv(output_file, index=False)
    print(f"\nSUKCES! Zapisano prawdziwe dane do: {output_file}")
    print(df.head())
else:
    print("\nNie udało się pobrać danych.")

--- Pobieranie fundamentów z Yahoo Finance dla 6 spółek ---
Pobieranie dla: AAPL...
   -> Pobrano dane.
Pobieranie dla: AMZN...
   -> Pobrano dane.
Pobieranie dla: NVDA...
   -> Pobrano dane.
Pobieranie dla: INTC...
   -> Pobrano dane.
Pobieranie dla: AMD...
   -> Pobrano dane.
Pobieranie dla: BRK-B...
   -> Pobrano dane.

SUKCES! Zapisano prawdziwe dane do: fundamentals_data/company_earnings.csv
  symbol report_date  year                ebitda            net_income  \
0   AAPL  2025-09-30  2025 144748000000.00000000 112010000000.00000000   
1   AAPL  2024-09-30  2024 134661000000.00000000  93736000000.00000000   
2   AAPL  2023-09-30  2023 125820000000.00000000  96995000000.00000000   
3   AAPL  2022-09-30  2022 130541000000.00000000  99803000000.00000000   
4   AAPL  2021-09-30  2021                   NaN                   NaN   

                revenue  
0 416161000000.00000000  
1 391035000000.00000000  
2 383285000000.00000000  
3 394328000000.00000000  
4                   NaN  

In [4]:
import yfinance as yf
import pandas as pd
import os
import time

# Lista Twoich spółek
SYMBOLS = [
    "AAPL", "AMZN", "NVDA", "INTC", "AMD", "BRK-B"
]

company_details = []

print(f"--- Pobieranie informacji o {len(SYMBOLS)} spółkach ---")

for sym in SYMBOLS:
    print(f"Pobieranie info dla: {sym}...")

    try:
        ticker = yf.Ticker(sym)

        # To jest ten "magiczny" słownik z setkami danych
        info = ticker.info

        # Wyciągamy to, co przyda się w analizie Big Data
        company_details.append({
            "symbol": sym.replace("-", "."), # Poprawka dla BRK.B
            "short_name": info.get("shortName"),
            "sector": info.get("sector", "N/A"),           # WAŻNE: Sektor
            "industry": info.get("industry", "N/A"),       # WAŻNE: Branża
            "market_cap": info.get("marketCap"),           # Wielkość firmy
            "employees": info.get("fullTimeEmployees"),    # Liczba pracowników
            "pe_ratio": info.get("trailingPE"),            # Cena do Zysku
            "beta": info.get("beta"),                      # Ryzyko
            "recommendation": info.get("recommendationKey"), # np. 'buy', 'hold'
            "currency": info.get("currency")
        })

        print(f"   -> Sukces.")

    except Exception as e:
        print(f"   ! Błąd dla {sym}: {e}")

    time.sleep(1)

# --- ZAPIS ---
os.makedirs("static_data", exist_ok=True)

if company_details:
    df = pd.DataFrame(company_details)
    output_file = "static_data/company_info.csv"
    df.to_csv(output_file, index=False)
    print(f"\nSUKCES! Zapisano dane do: {output_file}")
    # Pokaż podgląd
    print(df[['symbol', 'sector', 'market_cap', 'recommendation']])
else:
    print("\nNic nie pobrano.")

--- Pobieranie informacji o 6 spółkach ---
Pobieranie info dla: AAPL...
   -> Sukces.
Pobieranie info dla: AMZN...
   -> Sukces.
Pobieranie info dla: NVDA...
   -> Sukces.
Pobieranie info dla: INTC...
   -> Sukces.
Pobieranie info dla: AMD...
   -> Sukces.
Pobieranie info dla: BRK-B...
   -> Sukces.

SUKCES! Zapisano dane do: static_data/company_info.csv
  symbol              sector     market_cap recommendation
0   AAPL          Technology  3819990417408            buy
1   AMZN   Consumer Cyclical  2633534799872     strong_buy
2   NVDA          Technology  4512107986944     strong_buy
3   INTC          Technology   211239911424           hold
4    AMD          Technology   336939515904            buy
5  BRK.B  Financial Services  1073543708672            buy


In [5]:
import yfinance as yf
import pandas as pd
import os
import time

# --- KONFIGURACJA ---
# Lista symboli (Finnhub style)
SYMBOLS_FINNHUB = [
    "AAPL", "AMZN", "NVDA", "INTC", "AMD", "BRK.B",
    "BINANCE:BTCUSDT", "BINANCE:ETHUSDT" # Krypto też obsłużymy!
]

# Mapowanie na Yahoo Finance
SYMBOL_MAP = {
    "AAPL": "AAPL",
    "AMZN": "AMZN",
    "NVDA": "NVDA",
    "INTC": "INTC",
    "AMD": "AMD",
    "BRK.B": "BRK-B",
    "BINANCE:BTCUSDT": "BTC-USD",
    "BINANCE:ETHUSDT": "ETH-USD"
}

market_data = []

print("--- Pobieranie kontekstu rynkowego (High/Low/Targets) ---")

for finnhub_sym in SYMBOLS_FINNHUB:
    yahoo_sym = SYMBOL_MAP.get(finnhub_sym)
    print(f"Pobieranie dla: {finnhub_sym} -> {yahoo_sym}...")

    try:
        ticker = yf.Ticker(yahoo_sym)
        info = ticker.info

        market_data.append({
            "symbol": finnhub_sym,  # Używamy nazwy z Finnhub, żeby pasowała do Kafki!
            "year_high": info.get("fiftyTwoWeekHigh"),       # Szczyt roczny
            "year_low": info.get("fiftyTwoWeekLow"),         # Dołek roczny
            "avg_50d": info.get("fiftyDayAverage"),          # Średnia 50-dniowa
            "analyst_target": info.get("targetMeanPrice"),   # Cena docelowa (tylko dla akcji)
            "recommendation": info.get("recommendationKey")  # Np. 'buy', 'hold'
        })
        print("   -> OK")

    except Exception as e:
        print(f"   ! Błąd: {e}")

    time.sleep(1)

# --- ZAPIS ---
os.makedirs("static_data", exist_ok=True)
df = pd.DataFrame(market_data)
output_file = "static_data/market_context.csv"

# Wypełniamy braki (krypto nie ma "analyst_target", wpiszemy 0)
df = df.fillna(0)

df.to_csv(output_file, index=False)
print(f"\nGotowe! Zapisano w: {output_file}")
print(df.head())

--- Pobieranie kontekstu rynkowego (High/Low/Targets) ---
Pobieranie dla: AAPL -> AAPL...
   -> OK
Pobieranie dla: AMZN -> AMZN...
   -> OK
Pobieranie dla: NVDA -> NVDA...
   -> OK
Pobieranie dla: INTC -> INTC...
   -> OK
Pobieranie dla: AMD -> AMD...
   -> OK
Pobieranie dla: BRK.B -> BRK-B...
   -> OK
Pobieranie dla: BINANCE:BTCUSDT -> BTC-USD...
   -> OK
Pobieranie dla: BINANCE:ETHUSDT -> ETH-USD...
   -> OK

Gotowe! Zapisano w: static_data/market_context.csv
  symbol    year_high     year_low      avg_50d  analyst_target recommendation
0   AAPL 288.62000000 169.21000000 272.81360000    287.70682000            buy
1   AMZN 258.60000000 161.38000000 232.95860000    295.86450000     strong_buy
2   NVDA 212.19000000  86.62000000 186.69760000    252.28140000     strong_buy
3   INTC  44.99000000  17.67000000  38.21440000     38.30694000           hold
4    AMD 267.08000000  76.48000000 225.07380000    285.12045000            buy
